# Visión: detección de zonas del ticket (YOLO ajustado)
Entrenamiento del concurso. Correr en el servidor NVIDIA y conservar logs, pesos y evidencias.
**Declarar en la ficha:** modelo base, licencia, procedencia de los datos (reales / sintéticos) y que el OCR es preentrenado.
Las corridas largas se lanzan con `training/common/run_job.sh` dentro de `tmux`; este notebook sirve para humo, revisión de datos y evaluación.

In [ ]:
# 1) Entorno. Si torch.cuda.is_available() sale False, NO sigas: avisa antes de entrenar.
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:600])
print("cuda:", torch.cuda.is_available(), "| torch:", torch.__version__)
# %pip install ultralytics     # despues de instalar, vuelve a correr esta celda y verifica cuda=True

In [ ]:
# 2) Configuracion (UN solo lugar para cambiar parametros)
import os, subprocess
from pathlib import Path

# Jupyter ejecuta desde la carpeta del notebook: movemos el cwd a la raiz del repo para que las rutas funcionen
ROOT = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True).stdout.strip() or "."
os.chdir(ROOT); print("cwd:", os.getcwd())
"""DATA = Path("training/vision/data")      # training/vision/data/{images,labels}/{train,val,test}
CLASES = ["renglon", "cantidad", "precio", "total", "fecha", "tienda"]

BASE = "yolo11s.pt"                      # modelo base declarado
SMOKE = True                             # True = corrida de humo (1 epoca). False = principal
CFG = dict(epochs=1 if SMOKE else 80, imgsz=960, batch=8 if SMOKE else 16, patience=20, seed=0)
RUNS = Path("training/vision/runs"); NAME = "humo" if SMOKE else "principal"""

In [ ]:
# 3) Revision del dataset: conteos, labels faltantes, duplicados ENTRE conjuntos (fuga) y cajas por clase
import hashlib, collections
def h(p): return hashlib.md5(p.read_bytes()).hexdigest()
vistos, reporte = {}, {}
for sp in ["train", "val", "test"]:
    imgs = sorted((DATA/"images"/sp).glob("*.*"))
    sin_label = [i.name for i in imgs if not (DATA/"labels"/sp/f"{i.stem}.txt").exists()]
    cajas = collections.Counter()
    for l in (DATA/"labels"/sp).glob("*.txt"):
        for ln in l.read_text().splitlines():
            if ln.strip(): cajas[CLASES[int(ln.split()[0])]] += 1
    for i in imgs:
        k = h(i)
        if k in vistos and vistos[k] != sp: print("FUGA: duplicado en", vistos[k], "y", sp, i.name)
        vistos.setdefault(k, sp)
    reporte[sp] = {"imagenes": len(imgs), "sin_label": len(sin_label), "cajas": dict(cajas)}
    print(sp, reporte[sp])
import json; Path("training/vision").mkdir(parents=True, exist_ok=True)
Path("training/vision/dataset_report.json").write_text(json.dumps(reporte, indent=2))
# Nota: 'fuga' por mismo ticket fotografiado dos veces o recortes del mismo ticket no la detecta el hash. Dividan por ticket.

In [ ]:
# 4) data.yaml
import yaml
yml = DATA.resolve()/"data.yaml"
yml.write_text(yaml.dump({"path": str(DATA.resolve()), "train": "images/train", "val": "images/val",
                          "test": "images/test", "names": {i: n for i, n in enumerate(CLASES)}}))
print(yml.read_text())

In [ ]:
# 5) Entrenamiento. (Para la corrida principal larga: SMOKE=False y mejor lanzarla con run_job.sh en tmux)
import time, datetime, torch
from ultralytics import YOLO
t0 = time.time(); inicio = datetime.datetime.now().isoformat(timespec="seconds")
model = YOLO(BASE)
res = model.train(data=str(yml), device=0, project=str(RUNS), name=NAME, exist_ok=True,
                  # Aumentos pensados para TEXTO/TICKETS: sin volteos (espejan el texto), giro y perspectiva leves
                  fliplr=0.0, flipud=0.0, degrees=3.0, perspective=0.0005, shear=1.0,
                  scale=0.3, translate=0.05, hsv_h=0.01, hsv_s=0.4, hsv_v=0.4, **CFG)
log = {"inicio": inicio, "duracion_s": round(time.time()-t0), "gpu": torch.cuda.get_device_name(0),
       "modelo_base": BASE, "cfg": CFG, "carpeta": str(res.save_dir)}
Path(res.save_dir/"registro_entrenamiento.json").write_text(json.dumps(log, indent=2)); print(log)

In [ ]:
# 6) Evaluacion en TEST (reservado). Ojo: model.val() sin split evalua VALIDACION, no prueba.
best = Path(res.save_dir)/"weights"/"best.pt"
m = YOLO(str(best)).val(data=str(yml), split="test", imgsz=CFG["imgsz"], device=0)
out = {"mAP50": m.box.map50, "mAP50_95": m.box.map, "precision": m.box.mp, "recall": m.box.mr,
       "por_clase_mAP50": {CLASES[int(c)]: float(v) for c, v in zip(m.box.ap_class_index, m.box.ap50)}}
Path("training/vision/eval_test.json").write_text(json.dumps(out, indent=2, ensure_ascii=False)); print(out)
# Comparacion con el modelo base: yolo11s.pt no conoce estas clases, asi que no es comparable; la linea base es
# OCR de imagen completa + reglas (explicarlo en la ficha).

In [ ]:
# 7) Ejemplos de aciertos y errores (revisar a mano y elegir los que iran a la presentacion)
out_dir = Path("training/vision/ejemplos"); out_dir.mkdir(exist_ok=True)
for r in YOLO(str(best)).predict(source=str(DATA/"images"/"test"), imgsz=CFG["imgsz"], conf=0.25, device=0, stream=True):
    r.save(filename=str(out_dir/Path(r.path).name))
print("Guardados en", out_dir)

In [ ]:
# 8) Registrar hashes de los pesos (evidencia de que la app usa lo entrenado)
import subprocess
subprocess.run(["python", "training/common/sha256_weights.py", "training/vision/weights_manifest.json", BASE, str(best)], check=True)

In [ ]:
# 9) OPCIONAL: TensorRT (herramienta NVIDIA) y latencia. Si falla en la Spark, documenta el error y sigue con PyTorch.
import numpy as np
img = str(next((DATA/"images"/"test").glob("*.*")))
def lat(p, n=30):
    mm = YOLO(p); mm(img, imgsz=CFG["imgsz"], verbose=False)
    t = []
    for _ in range(n):
        s = time.time(); mm(img, imgsz=CFG["imgsz"], verbose=False); t.append((time.time()-s)*1000)
    return float(np.median(t))
res_lat = {"pytorch_ms": lat(str(best))}
try:
    eng = YOLO(str(best)).export(format="engine", half=True, imgsz=CFG["imgsz"], device=0)
    res_lat["tensorrt_ms"] = lat(eng)
except Exception as e:
    res_lat["tensorrt_error"] = str(e)
Path("training/vision/latencia.json").write_text(json.dumps(res_lat, indent=2)); print(res_lat)